# 06 · Final model: YOLOv8m @1024 trained on train + val
Public LB showed that inference tricks tuned on the 7 val clips did not transfer to test (0.203 → 0.196). The 640 run also peaked after 1 epoch, i.e. generalisation across clips is the bottleneck. So the final model uses a larger backbone (YOLOv8m) and all 63 labelled clips (every 3rd frame). No validation is left, so the epoch count is fixed in advance and the model is judged by the public LB only.

In [1]:
# === Colab only: run this cell first (downloads data, keeps outputs on Google Drive) ===
import os, glob, subprocess
from getpass import getpass
from google.colab import drive
drive.mount('/content/drive')
WORKDIR = '/content/drive/MyDrive/hw2_runs'  # results/, images/, weights/ survive disconnects here
os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)

if not glob.glob('/content/data/**/test_images.csv', recursive=True):
    subprocess.run(['pip', 'install', '-q', '-U', 'kaggle'], check=True)
    if not os.environ.get('KAGGLE_API_TOKEN'):
        os.environ['KAGGLE_API_TOKEN'] = getpass('Paste Kaggle API token (KGAT_...): ').strip()
    subprocess.run(['kaggle', 'competitions', 'download', '-c',
                    'multiple-object-video-detection-hw-2-ucu-ai-2026', '-p', '/content/data'], check=True)
    for z in glob.glob('/content/data/*.zip'):
        subprocess.run(['unzip', '-q', '-o', z, '-d', '/content/data'], check=True); os.remove(z)

os.environ['DATA_ROOT'] = os.path.dirname(glob.glob('/content/data/**/test_images.csv', recursive=True)[0])
print('DATA_ROOT =', os.environ['DATA_ROOT'])
# repo copy on Drive may be stale: always use the latest src/common.py from GitHub
subprocess.run('rm -rf repo', shell=True)

Mounted at /content/drive
Paste Kaggle API token (KGAT_...): ··········
DATA_ROOT = /content/data


CompletedProcess(args='rm -rf repo', returncode=0)

## 0. Setup

In [2]:
import os, sys, subprocess, shutil
from pathlib import Path
REPO_URL = 'https://github.com/Makaron4ik23/hw2.git'
if not any(Path(d, 'common.py').exists() for d in ['src', '../src', 'repo/src']):
    subprocess.run(['git', 'clone', '-q', REPO_URL, 'repo'], check=True)
sys.path.insert(0, next(d for d in ['src', '../src', 'repo/src'] if Path(d, 'common.py').exists()))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.208', 'pycocotools'], check=True)
import torch
from ultralytics import YOLO
from common import *
seed_everything()
ROOT = find_data_root()
TMP = Path('/tmp/hw2')
OUT = Path('results'); OUT.mkdir(exist_ok=True)
WEIGHTS = Path('weights'); WEIGHTS.mkdir(exist_ok=True)
print(torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Tesla T4


## 1. Data: same masked copy (every 5th frame), train + val clips together

In [3]:
STEP, EPOCHS, IMGSZ, NAME = 3, 10, 1024, 'yolo_m_1024_trainval'  # ~5h GPU budget: every 3rd frame
yaml_path = prepare_training_copy(ROOT, TMP / 'data_s3', step_train=STEP, step_val=STEP)
tv_yaml = TMP / 'data_s3' / 'data_trainval.yaml'
tv_yaml.write_text(yaml_path.read_text().replace('train: train/images', 'train: [train/images, val/images]'))
print(tv_yaml.read_text())

train: 8090 frames (every 3th)
val: 951 frames (every 3th)
path: /tmp/hw2/data_s3
train: [train/images, val/images]
val: val/images
nc: 8
names: ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]



## 2. Train (val metric printed during training is not held-out: val clips are in the training set)

In [4]:
model = YOLO('yolov8m.pt')
model.train(data=str(tv_yaml), name=NAME, epochs=EPOCHS, imgsz=IMGSZ, batch=6, seed=SEED, deterministic=True,
            cos_lr=True, close_mosaic=2, workers=4, project=str(Path('runs').resolve()), exist_ok=True,
            verbose=False, plots=True, device=0, save_period=1)  # runs/ is on Drive: checkpoints survive a disconnect
run_dir = Path(model.trainer.save_dir)
shutil.copy(run_dir / 'weights' / 'last.pt', WEIGHTS / f'{NAME}.pt')  # last epoch: no held-out val to pick 'best'
shutil.copy(run_dir / 'results.csv', OUT / f'ultra_{NAME}.csv')

New https://pypi.org/project/ultralytics/8.4.213 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=2, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/tmp/hw2/data_s3/data_trainval.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, mom

PosixPath('results/ultra_yolo_m_1024_trainval.csv')

## 3. Submission (plain inference — the setting that scored best on the public LB)

In [5]:
model = YOLO(WEIGHTS / f'{NAME}.pt')
test_preds, fps = predict_ultralytics(model, test_frames(ROOT), imgsz=IMGSZ)
write_submission(test_preds, ROOT, f'submission_{NAME}.csv', min_score=0.01)
print('test fps:', fps)

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.